# El costo invisible del progreso
### ¿Cambia la identificación de los países que desacoplan su crecimiento de las emisiones de CO₂ al pasar de la contabilidad territorial a la basada en consumo?

**Proyecto 2 · DATA-1002 Aplicaciones en Ciencia de Datos · Universidad de los Andes**

Integrantes: _(completar)_

| Contenido | Sesión del curso |
|---|---|
| 1. Comprensión del problema (5W, objetivo, pregunta, hipótesis, requerimientos) | 2 |
| 2. Librerías y estructura del proyecto | 3 |
| 3. Fuentes y carga de datos | 9 |
| 4. Revisión inicial y diccionario de datos | 4 |
| 5. Calidad de los datos | 8 |
| 6. Limpieza y transformación | 5 |
| 7. Integración de las fuentes | 10 |
| 8. Variables derivadas | 8 |
| 9. Análisis exploratorio | 6 y 7 |
| 10. Clasificación del desacoplamiento y respuesta a la pregunta | 6 y 7 |
| 11. Desagregación, robustez y sesgos | 7 y 11 |
| 12. Hallazgos | 6 y 7 |
| 13. Exportación para el dashboard | 14 |
| 14. Limitaciones, próximos pasos y fuentes | 3 y 11 |

## 1. Comprensión del problema

### Las 5W

| Pregunta | Respuesta |
|---|---|
| **¿Quién?** | Comité técnico de desarrollo sostenible y transición climática (usuario). Afecta a gobiernos y a la población de países que exportan o importan bienes intensivos en carbono. |
| **¿Qué?** | Saber qué países crecen sin aumentar sus emisiones de CO₂ y si ese resultado se sostiene cuando se cuentan las emisiones del comercio internacional. |
| **¿Dónde?** | Países con datos de emisiones basadas en consumo (unos 120). |
| **¿Cuándo?** | 1990–2023, período común de las cuatro fuentes. |
| **¿Por qué?** | Si solo se mira la contabilidad territorial, un país puede parecer exitoso aunque haya trasladado parte de su producción contaminante a otros países. |

### Objetivo de negocio
Apoyar al comité en la selección de **países referentes** (su desacoplamiento se sostiene con ambas contabilidades) y de **países que requieren diagnóstico adicional** (su avance territorial no se sostiene al incluir el comercio).

**Criterios de éxito del proyecto:**
- Clasificar a cada país con ambas contabilidades usando una regla explícita y reproducible.
- Cuantificar cuántos países cambian de clase y cuáles son.
- Entregar un dashboard alimentado por los datos integrados que muestre la pregunta, los indicadores, los hallazgos y al menos una limitación.

### Pregunta analítica
¿En qué medida cambia la identificación de países que han logrado desacoplar su crecimiento económico de las emisiones de CO₂ al comparar las emisiones territoriales con las asociadas al consumo?

### Hipótesis
Una parte de los países que muestran desacoplamiento absoluto con la contabilidad territorial pierde esa clasificación con la contabilidad de consumo, y esto ocurre con más frecuencia en países **importadores netos** de emisiones.

### Requerimientos analíticos
- **Datos:** PIB real, CO₂ territorial, CO₂ basado en consumo y población, por país y año, con una llave común (código ISO-3 + año).
- **Accesibilidad:** datos abiertos de Our World in Data, licencia CC BY 4.0.
- **Métrica de éxito técnico:** unión 1 a 1 sin duplicados ni pérdidas inexplicadas; clasificación reproducible; resultado robusto a otro período.

## 2. Librerías y estructura del proyecto

Estructura del repositorio (Sesión 3):

```
Proyecto 2/
├── README.md
├── requirements.txt
├── data/
│   ├── raw/          ← datos originales, no se editan
│   └── processed/    ← datos integrados que alimentan el dashboard
├── notebooks/
│   └── proyecto2.ipynb
└── outputs/
    └── figures/
```

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Funciona si el notebook se ejecuta desde la raíz del proyecto o desde notebooks/
RAIZ = Path(".") if Path("data/raw").exists() else Path("..")
RAW = RAIZ / "data" / "raw"
PROCESSED = RAIZ / "data" / "processed"
FIGURAS = RAIZ / "outputs" / "figures"
PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

print("Archivos en data/raw:", sorted(p.name for p in RAW.iterdir()))

## 3. Fuentes y carga de datos

| Fuente | Productor original | Acceso | Fecha de consulta | Licencia |
|---|---|---|---|---|
| CO₂ territorial per cápita | Global Carbon Budget (2026), procesado por Our World in Data | [ourworldindata.org/co2-emissions](https://ourworldindata.org/co2-emissions) (descarga ZIP) | 22/11/2026 | CC BY 4.0 |
| CO₂ basado en consumo per cápita | Global Carbon Budget (2026), procesado por Our World in Data | [ourworldindata.org/co2-emissions](https://ourworldindata.org/co2-emissions) (descarga ZIP) | 22/11/2026 | CC BY 4.0 |
| PIB real, PPA (dólares internacionales de 2021) | Banco Mundial, Eurostat, OCDE y FMI (2026), vía Our World in Data | [ourworldindata.org/grapher/gdp-worldbank](https://ourworldindata.org/grapher/gdp-worldbank) (descarga ZIP) | 22/11/2026 | CC BY 4.0 |
| Población | HYDE (2023), Gapminder (2022), UN WPP (2024), vía Our World in Data | [ourworldindata.org/grapher/population](https://ourworldindata.org/grapher/population) (`requests`) | 22/11/2026 | CC BY 4.0 |

**¿Por qué cuatro fuentes?** Las dos series de CO₂ vienen *per cápita* y el PIB viene en *total*. La población permite convertir las emisiones a totales y compararlas con el crecimiento del PIB.

Los archivos de OWID vienen en ZIP con el CSV, un `readme.md` y un `.metadata.json`; se leen directamente desde el ZIP, sin descomprimir a mano, para no tocar los datos originales.

In [ ]:
def leer_owid_zip(ruta_zip):
    """Lee el CSV que viene dentro de un ZIP descargado de Our World in Data."""
    with zipfile.ZipFile(ruta_zip) as z:
        nombre_csv = [n for n in z.namelist() if n.endswith(".csv")][0]
        with z.open(nombre_csv) as f:
            return pd.read_csv(f)

co2_terr = leer_owid_zip(RAW / "co2-emissions-per-capita.zip")
co2_cons = leer_owid_zip(RAW / "per-capita-consumption-based-co2-emissions.zip")
pib = leer_owid_zip(RAW / "gdp-worldbank.zip")

La población se consulta con `requests` (Sesión 9) solo si el archivo no existe, y se guarda en `data/raw/population.csv` para no repetir la consulta. El archivo incluido contiene los 120 países con datos de consumo, 1990–2023.

In [ ]:
RUTA_POBLACION = RAW / "population.csv"
URL_POBLACION = "https://ourworldindata.org/grapher/population.csv?v=1&csvType=full&useColumnShortNames=false"

if not RUTA_POBLACION.exists():
    import requests
    r = requests.get(URL_POBLACION, timeout=60)
    print("Estado de la consulta:", r.status_code)
    r.raise_for_status()
    RUTA_POBLACION.write_bytes(r.content)

poblacion = pd.read_csv(RUTA_POBLACION)

## 4. Revisión inicial y diccionario de datos

**Granularidad:** en las cuatro fuentes, **una fila = una entidad (país o agregado) en un año**.

### Diccionario de datos (fuentes originales)

| Columna | Tipo analítico | Significado | Unidad | Fuente |
|---|---|---|---|---|
| `Entity` | Nominal | Nombre del país o agregado | — | Todas |
| `Code` | Identificador | Código ISO alfa-3 (los agregados usan `OWID_*`, `WB_*` o vienen vacíos) | — | Todas |
| `Year` | Numérica discreta (tiempo) | Año de la observación | año | Todas |
| `CO₂ emissions per capita` | Numérica continua | CO₂ fósil e industrial emitido dentro del país, por persona | t CO₂ por persona | CO₂ territorial |
| `Per capita consumption-based CO₂ emissions` | Numérica continua | CO₂ asociado a lo que consume el país (territorial − exportaciones + importaciones), por persona | t CO₂ por persona | CO₂ consumo |
| `GDP` | Numérica continua | PIB real en paridad de poder adquisitivo | dólares internacionales de 2021 | PIB |
| `Population` | Numérica discreta | Población a mitad de año | personas | Población |

In [ ]:
fuentes = {"CO₂ territorial": co2_terr, "CO₂ consumo": co2_cons, "PIB": pib, "Población": poblacion}

for nombre, df in fuentes.items():
    print(f"===== {nombre} =====")
    print("Dimensiones:", df.shape)
    df.info()
    print(df.head(3), "\n")

In [ ]:
def diagnostico(nombre, df):
    """Diagnóstico mínimo de la Sesión 4: tamaño, faltantes, duplicados y cardinalidad."""
    valor = df.columns[-1]
    codigos = df["Code"].astype("string").fillna("")
    return {
        "fuente": nombre,
        "filas": df.shape[0],
        "años": f"{df['Year'].min()}–{df['Year'].max()}",
        "entidades (nunique)": df["Code"].nunique(dropna=False),
        "faltantes en Code": int(df["Code"].isna().sum()),
        "faltantes en el valor": int(df[valor].isna().sum()),
        "filas duplicadas": int(df.duplicated().sum()),
        "duplicados entidad-año": int(df.duplicated(["Entity", "Year"]).sum()),
        "agregados (OWID_ / WB_)": int(codigos.str.match(r"^(OWID_|WB_)").sum()),
        "valor mínimo": df[valor].min(),
    }

pd.DataFrame([diagnostico(n, d) for n, d in fuentes.items()]).set_index("fuente")

## 5. Calidad de los datos (Sesión 8)

| Dimensión | Revisión | Resultado |
|---|---|---|
| **Completitud** | Faltantes en el valor | No hay faltantes; las series sí difieren en cobertura de años y países (consumo: 1990–2025 y ~120 países). |
| **Unicidad** | Duplicados entidad-año | Ninguno en las cuatro fuentes. |
| **Validez** | Valores negativos o cero | Ningún valor negativo (ver `valor mínimo`). |
| **Consistencia** | Mezcla de países y agregados | Hay continentes, grupos de ingreso y regiones del Banco Mundial (`WB_LAC`, etc.) junto a los países: si se dejan, un país se contaría dos veces. |
| **Exactitud** | ¿El dato representa lo que dice medir? | En algunos países la serie de consumo es una **copia** de la territorial (se revisa abajo). |

In [ ]:
# Consistencia: ¿qué entidades no son países?
no_paises = pd.concat([
    d.loc[~d["Code"].astype("string").fillna("").str.match(r"^[A-Z]{3}$"), ["Entity", "Code"]]
    for d in fuentes.values()
]).drop_duplicates()
print(f"Entidades que no son países: {len(no_paises)}")
no_paises.head(15)

### Exactitud: emisiones de consumo que son copia de las territoriales

Si el CO₂ de consumo es **idéntico** al territorial en un país-año, el Global Carbon Budget no tenía datos de comercio para ese año y repitió el valor territorial. No es un resultado real: si se usa, el "cambio" de las emisiones de consumo de ese país sería un artefacto.

In [ ]:
comparar = co2_terr.merge(co2_cons, on=["Code", "Year"], suffixes=("_terr", "_cons"))
comparar = comparar[comparar["Code"].astype("string").fillna("").str.match(r"^[A-Z]{3}$")]
iguales = comparar[np.isclose(comparar.iloc[:, 3], comparar.iloc[:, 5], rtol=0, atol=1e-9)]

copias = iguales.groupby(["Code", "Entity_terr"])["Year"].agg(desde="min", hasta="max", años="count")
copias

**Decisión:** estos países no se clasifican con emisiones de consumo cuando alguna copia cae dentro del período inicial de comparación. Quedan marcados como *Sin dato confiable de consumo*; su clasificación territorial se conserva. No se imputan ni se borran: se documentan (Sesión 8, "limpiar no es maquillar").

## 6. Limpieza y transformación (Sesión 5)

Decisiones, en orden:
1. **Filtrar:** conservar solo países (código ISO alfa-3 de tres letras mayúsculas).
2. **Seleccionar y renombrar:** columnas con nombres cortos en español.
3. **Registrar la pérdida de datos:** cuántas filas se excluyen en cada paso.

In [ ]:
ES_PAIS = r"^[A-Z]{3}$"
bitacora = []

def solo_paises(df, nombre_fuente, nombre_valor):
    codigos = df["Code"].astype("string").fillna("")
    salida = df[codigos.str.match(ES_PAIS)].copy()
    bitacora.append({"fuente": nombre_fuente, "filas originales": len(df),
                     "filas de países": len(salida), "filas excluidas (agregados)": len(df) - len(salida)})
    return salida.rename(columns={"Entity": "pais", "Code": "iso3", "Year": "anio", salida.columns[-1]: nombre_valor})

terr = solo_paises(co2_terr, "CO₂ territorial", "co2_terr_pc")
cons = solo_paises(co2_cons, "CO₂ consumo", "co2_cons_pc").drop(columns="pais")
gdp = solo_paises(pib, "PIB", "pib").drop(columns="pais")
pob = solo_paises(poblacion, "Población", "poblacion").drop(columns="pais")

pd.DataFrame(bitacora).set_index("fuente")

## 7. Integración de las fuentes (Sesión 10)

**Checklist antes de unir:**

| Pregunta | Respuesta |
|---|---|
| ¿Qué representa cada fila? | Un país en un año, en las cuatro tablas. |
| ¿Cuál es la llave? | Compuesta: `iso3` + `anio`, con el mismo formato en todas. |
| ¿Hay duplicados en la llave? | No (se verifica con `assert`). |
| ¿Qué cardinalidad se espera? | 1 a 1. Se valida con `validate="one_to_one"`. |
| ¿Qué tipo de unión? | `outer` con `indicator=True` para auditar, e `inner` para el panel final: el análisis exige las cuatro variables. |
| ¿Qué pasa con los registros sin pareja? | Se listan y se reportan como limitación de cobertura. |

In [ ]:
for nombre, df in [("territorial", terr), ("consumo", cons), ("PIB", gdp), ("población", pob)]:
    assert not df.duplicated(["iso3", "anio"]).any(), f"Llave duplicada en {nombre}"

# Auditoría: consumo (la fuente más restrictiva) frente a PIB
auditoria = cons.merge(gdp, on=["iso3", "anio"], how="outer", validate="one_to_one", indicator=True)
print(auditoria["_merge"].value_counts().rename({"left_only": "solo consumo", "right_only": "solo PIB", "both": "ambos"}))
print("\nPaíses con emisiones de consumo pero sin PIB:",
      sorted(auditoria.loc[auditoria["_merge"] == "left_only", "iso3"].unique()))

In [ ]:
paso1 = terr.merge(cons, on=["iso3", "anio"], how="inner", validate="one_to_one")
paso2 = paso1.merge(gdp, on=["iso3", "anio"], how="inner", validate="one_to_one")
panel = paso2.merge(pob, on=["iso3", "anio"], how="inner", validate="one_to_one")

pd.DataFrame({
    "filas": [len(terr), len(paso1), len(paso2), len(panel)],
    "países": [terr["iso3"].nunique(), paso1["iso3"].nunique(), paso2["iso3"].nunique(), panel["iso3"].nunique()],
}, index=["CO₂ territorial", "+ CO₂ consumo", "+ PIB", "+ población (panel final)"])

In [ ]:
assert not panel.duplicated(["iso3", "anio"]).any()
assert panel[["co2_terr_pc", "co2_cons_pc", "pib", "poblacion"]].notna().all().all()

print("Panel integrado:", panel.shape)
print("Países:", panel["iso3"].nunique(), "| Años:", panel["anio"].min(), "–", panel["anio"].max())
print("Observaciones por país:", panel.groupby("iso3").size().describe()[["min", "50%", "max"]].to_dict())

**Resultado:** la unión no duplicó filas (`validate` no falló) y cada fila sigue siendo un país-año. El año final es 2023 porque la serie de población llega hasta ese año. Taiwán y Venezuela quedan fuera por no tener PIB del Banco Mundial.

## 8. Variables derivadas

| Variable | Fórmula | Unidad | Para qué sirve |
|---|---|---|---|
| `co2_terr_mt` | CO₂ territorial per cápita × población / 10⁶ | millones de t CO₂ | Comparar emisiones totales con el PIB total |
| `co2_cons_mt` | CO₂ de consumo per cápita × población / 10⁶ | millones de t CO₂ | Ídem, con contabilidad de consumo |
| `pib_pc` | PIB / población | dólares internacionales de 2021 por persona | Clasificación per cápita y grupos de ingreso |
| `intensidad_terr`, `intensidad_cons` | CO₂ / PIB | kg CO₂ por dólar | Eficiencia en carbono de la economía |
| `brecha_cons_pct` | (consumo − territorial) / territorial × 100 | % | Peso del comercio en las emisiones |
| `balance_comercio` | "Importador neto" si consumo > territorial | categoría | Hipótesis del proyecto |
| `consumo_es_copia` | consumo = territorial en ese país-año | booleano | Control de exactitud (sección 5) |

In [ ]:
panel = panel.assign(
    co2_terr_mt=panel["co2_terr_pc"] * panel["poblacion"] / 1e6,
    co2_cons_mt=panel["co2_cons_pc"] * panel["poblacion"] / 1e6,
    pib_pc=panel["pib"] / panel["poblacion"],
)
panel = panel.assign(
    intensidad_terr=panel["co2_terr_mt"] * 1e9 / panel["pib"],
    intensidad_cons=panel["co2_cons_mt"] * 1e9 / panel["pib"],
    brecha_cons_pct=(panel["co2_cons_pc"] - panel["co2_terr_pc"]) / panel["co2_terr_pc"] * 100,
    balance_comercio=np.where(panel["co2_cons_pc"] > panel["co2_terr_pc"], "Importador neto", "Exportador neto"),
    consumo_es_copia=np.isclose(panel["co2_cons_pc"], panel["co2_terr_pc"], rtol=0, atol=1e-9),
)

# Validación: las variables derivadas tienen sentido
assert (panel[["co2_terr_mt", "co2_cons_mt", "pib_pc"]] > 0).all().all()
panel.sort_values(["iso3", "anio"]).head()

## 9. Análisis exploratorio (Sesiones 6 y 7)

### 9.1 Distribución de las variables en 2023

In [ ]:
ultimo = panel[panel["anio"] == 2023]
ultimo[["co2_terr_pc", "co2_cons_pc", "pib_pc", "intensidad_terr", "brecha_cons_pct"]].describe(percentiles=[0.25, 0.5, 0.75])

In [ ]:
print("Balance de emisiones por comercio en 2023:")
print(ultimo["balance_comercio"].value_counts())
print(ultimo["balance_comercio"].value_counts(normalize=True).mul(100).round(1).astype(str) + " %")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(ultimo["brecha_cons_pct"].clip(-100, 200), bins=30, color="#2a9d8f", edgecolor="white")
ax.axvline(0, color="black", lw=1)
ax.axvline(ultimo["brecha_cons_pct"].median(), color="#e76f51", ls="--",
           label=f"Mediana: {ultimo['brecha_cons_pct'].median():.0f} %")
ax.set_title("En 2023, la mayoría de países consume más CO₂ del que produce en su territorio")
ax.set_xlabel("Brecha consumo − territorial (% de las emisiones territoriales; recortada a [−100, 200])")
ax.set_ylabel("Número de países")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURAS / "01_brecha_2023.png", dpi=150)
plt.show()

**Lectura:** la mayoría de países son importadores netos de emisiones (consumo > territorial). Los exportadores netos son sobre todo economías que producen bienes para el resto del mundo o exportan energía. Como la distribución es asimétrica, se reporta la mediana además de la media.

## 10. Clasificación del desacoplamiento

Se comparan dos ventanas de cinco años para no depender de un año atípico (crisis de 2008–2009, pandemia de 2020):

- **Período inicial:** promedio 1995–1999
- **Período final:** promedio 2019–2023

Para cada país se calcula el cambio porcentual del PIB y de las emisiones totales entre ambos períodos y se clasifica (definición usual de la OCDE):

| Clase | Regla |
|---|---|
| Desacoplamiento absoluto | El PIB crece y las emisiones caen |
| Desacoplamiento relativo | El PIB crece y las emisiones crecen menos que el PIB |
| Sin desacoplamiento | El PIB crece y las emisiones crecen igual o más que el PIB |
| PIB sin crecimiento | El PIB no creció (fuera del alcance de la pregunta) |

La clasificación se hace dos veces: con emisiones territoriales y con emisiones de consumo.

In [ ]:
INICIO = (1995, 1999)
FINAL = (2019, 2023)
SIN_DATO = "Sin dato confiable de consumo"
CLASES = ["Desacoplamiento absoluto", "Desacoplamiento relativo", "Sin desacoplamiento", "PIB sin crecimiento"]

def clasificar(g_pib, g_co2):
    if g_pib <= 0:
        return "PIB sin crecimiento"
    if g_co2 < 0:
        return "Desacoplamiento absoluto"
    if g_co2 < g_pib:
        return "Desacoplamiento relativo"
    return "Sin desacoplamiento"

def tabla_cambios(inicio, final, columnas=("pib", "co2_terr_mt", "co2_cons_mt")):
    """Promedia cada ventana y calcula el cambio % entre ambas. Exige los 5 años en cada ventana."""
    def ventana(a, b):
        v = panel[panel["anio"].between(a, b)]
        completos = v.groupby("iso3")["anio"].nunique()
        v = v[v["iso3"].isin(completos[completos == (b - a + 1)].index)]
        return v.groupby("iso3")[list(columnas)].mean()

    v0, v1 = ventana(*inicio), ventana(*final)
    comunes = v0.index.intersection(v1.index)
    cambio = (v1.loc[comunes] / v0.loc[comunes] - 1) * 100
    cambio.columns = ["g_" + c for c in columnas]
    nombres = panel.drop_duplicates("iso3").set_index("iso3")["pais"]
    cambio.insert(0, "pais", nombres.loc[comunes])
    return cambio

def marcar_sin_dato(tabla, inicio, columna):
    """Anula la clase de consumo de los países con datos copiados en la ventana inicial."""
    malos = panel.loc[panel["anio"].between(*inicio) & panel["consumo_es_copia"], "iso3"].unique()
    tabla.loc[tabla.index.isin(malos), columna] = SIN_DATO
    return tabla

cambios = tabla_cambios(INICIO, FINAL)
cambios["clase_territorial"] = [clasificar(p, c) for p, c in zip(cambios["g_pib"], cambios["g_co2_terr_mt"])]
cambios["clase_consumo"] = [clasificar(p, c) for p, c in zip(cambios["g_pib"], cambios["g_co2_cons_mt"])]
cambios = marcar_sin_dato(cambios, INICIO, "clase_consumo")

validos = cambios[cambios["clase_consumo"] != SIN_DATO].copy()
validos["cambia_de_clase"] = validos["clase_territorial"] != validos["clase_consumo"]
cambios["cambia_de_clase"] = validos["cambia_de_clase"]

print("Países en el panel:", panel["iso3"].nunique())
print("Sin los 5 años de la ventana inicial (no se clasifican):", sorted(set(panel["iso3"]) - set(cambios.index)))
print("Con consumo copiado en la ventana inicial:", sorted(cambios.index[cambios["clase_consumo"] == SIN_DATO]))
print("Países con ambas clasificaciones válidas:", len(validos))

In [ ]:
resumen = pd.DataFrame({
    "Territorial": validos["clase_territorial"].value_counts(),
    "Consumo": validos["clase_consumo"].value_counts(),
}).reindex(CLASES).fillna(0).astype(int)
resumen.loc["Total"] = resumen.sum()
resumen

### 10.1 Respuesta a la pregunta: ¿cuántos países cambian de clase?

La diagonal de la tabla cruzada son los países cuya clase se mantiene; lo que está fuera de ella son los que cambian.

In [ ]:
cruce = pd.crosstab(validos["clase_territorial"], validos["clase_consumo"],
                    rownames=["Clase territorial"], colnames=["Clase de consumo"])
cruce = cruce.reindex(index=CLASES, columns=CLASES, fill_value=0)

con_crecimiento = validos[validos["clase_territorial"] != "PIB sin crecimiento"]
n_cambian = int(con_crecimiento["cambia_de_clase"].sum())
abs_terr = int((con_crecimiento["clase_territorial"] == "Desacoplamiento absoluto").sum())
abs_cons = int((con_crecimiento["clase_consumo"] == "Desacoplamiento absoluto").sum())

print(f"Países con crecimiento del PIB: {len(con_crecimiento)}")
print(f"Cambian de clase al usar consumo: {n_cambian} ({n_cambian / len(con_crecimiento):.0%})")
print(f"Desacoplamiento absoluto: {abs_terr} con territorial → {abs_cons} con consumo")
cruce

In [ ]:
etiquetas = ["Absoluto", "Relativo", "Sin desacoplamiento"]
m = cruce.loc[CLASES[:3], CLASES[:3]].values

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(m, cmap="Blues")
ax.set_xticks(range(3), etiquetas)
ax.set_yticks(range(3), etiquetas)
for i in range(3):
    for j in range(3):
        ax.text(j, i, m[i, j], ha="center", va="center", fontsize=14,
                color="white" if m[i, j] > m.max() / 2 else "black")
ax.set_xlabel("Clase con emisiones de consumo")
ax.set_ylabel("Clase con emisiones territoriales")
ax.set_title(f"{n_cambian} de {len(con_crecimiento)} países cambian de clase al contar el comercio\n"
             f"({INICIO[0]}–{INICIO[1]} vs {FINAL[0]}–{FINAL[1]}; fuera de la diagonal = cambia)")
plt.tight_layout()
fig.savefig(FIGURAS / "02_tabla_cruzada.png", dpi=150)
plt.show()

### 10.2 Casos para el comité

- **Referentes:** desacoplamiento absoluto con ambas contabilidades.
- **Requieren diagnóstico:** desacoplamiento absoluto en territorio, pero no en consumo. Su reducción territorial no se sostiene al sumar las emisiones importadas.

In [ ]:
cols = ["pais", "g_pib", "g_co2_terr_mt", "g_co2_cons_mt", "clase_territorial", "clase_consumo"]
es_abs_terr = validos["clase_territorial"] == "Desacoplamiento absoluto"
es_abs_cons = validos["clase_consumo"] == "Desacoplamiento absoluto"

referentes = validos[es_abs_terr & es_abs_cons]
diagnostico_adicional = validos[es_abs_terr & ~es_abs_cons]

print(f"Referentes (absoluto en ambas): {len(referentes)}")
print(f"Requieren diagnóstico (absoluto solo en territorio): {len(diagnostico_adicional)}")
diagnostico_adicional[cols].sort_values("g_co2_cons_mt", ascending=False)

In [ ]:
referentes[cols].sort_values("g_co2_cons_mt")

### 10.3 Relación entre ambas contabilidades

Se grafica el cociente *período final / período inicial* en escala logarítmica: así caben en el mismo gráfico países que duplicaron o multiplicaron por 20 sus emisiones y países que las redujeron, sin recortar ninguno. Un valor de 1 significa que no cambiaron.

In [ ]:
corr_spearman = validos[["g_co2_terr_mt", "g_co2_cons_mt"]].corr(method="spearman").iloc[0, 1]
corr_pearson = validos[["g_co2_terr_mt", "g_co2_cons_mt"]].corr().iloc[0, 1]
print(f"Correlación entre el cambio territorial y el de consumo: Spearman {corr_spearman:.2f} | Pearson {corr_pearson:.2f}")

In [ ]:
colores = {"Desacoplamiento absoluto": "#2a9d8f", "Desacoplamiento relativo": "#e9c46a",
           "Sin desacoplamiento": "#e76f51", "PIB sin crecimiento": "#999999"}
ratio_t = 1 + validos["g_co2_terr_mt"] / 100
ratio_c = 1 + validos["g_co2_cons_mt"] / 100

fig, ax = plt.subplots(figsize=(8.5, 7))
for clase, idx in validos.groupby("clase_territorial").groups.items():
    ax.scatter(ratio_t[idx], ratio_c[idx], s=32, alpha=0.85, color=colores[clase], label=f"{clase} (territorial)")
lim = [0.35, 30]
ax.plot(lim, lim, color="grey", lw=1, ls="--", label="Igual cambio en ambas")
ax.axhline(1, color="black", lw=0.8)
ax.axvline(1, color="black", lw=0.8)
for iso in diagnostico_adicional.index:
    ax.annotate(iso, (ratio_t[iso], ratio_c[iso]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("CO₂ territorial: período final / período inicial (escala log)")
ax.set_ylabel("CO₂ de consumo: período final / período inicial (escala log)")
ax.set_title("Los países que reducen su CO₂ territorial no siempre reducen el de consumo\n"
             "(arriba de la diagonal: el consumo creció más que la producción doméstica)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
fig.savefig(FIGURAS / "03_territorial_vs_consumo.png", dpi=150)
plt.show()

**Lectura:** las dos contabilidades están muy correlacionadas (Spearman ≈ 0,9): en la mayoría de países ambas cuentan la misma historia. Las diferencias están en el cuadrante superior izquierdo (etiquetados): países que redujeron su CO₂ territorial mientras el de consumo creció.

### 10.4 Valores atípicos (Sesión 7)

Se identifican con el rango intercuartílico (IQR). Antes de decidir qué hacer, se revisa si son errores o casos reales.

In [ ]:
def atipicos_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return serie[(serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)]

for col in ["g_pib", "g_co2_terr_mt", "g_co2_cons_mt"]:
    at = atipicos_iqr(con_crecimiento[col])
    print(f"{col}: {len(at)} atípicos → {', '.join(f'{i} ({v:,.0f} %)' for i, v in at.sort_values(ascending=False).items())}")

**Decisión:** los atípicos son casos reales, no errores: economías de ingreso bajo que partieron de emisiones muy bajas en los años 90 (Etiopía, Camboya, Laos, Mozambique…) o que crecieron muy rápido (China, Catar). **Se conservan**, porque la clasificación usa umbrales (0 y el crecimiento del PIB) y no promedios, así que no la distorsionan. Por eso el gráfico usa escala logarítmica y las tablas reportan medianas.

In [ ]:
def trayectoria(iso, ax):
    d = panel[(panel["iso3"] == iso) & (panel["anio"] >= 1995)].set_index("anio")
    base = d.loc[1995]
    ax.plot(d.index, d["pib"] / base["pib"] * 100, label="PIB", color="#264653", lw=2)
    ax.plot(d.index, d["co2_terr_mt"] / base["co2_terr_mt"] * 100, label="CO₂ territorial", color="#e76f51")
    ax.plot(d.index, d["co2_cons_mt"] / base["co2_cons_mt"] * 100, label="CO₂ consumo", color="#2a9d8f", ls="--")
    ax.axhline(100, color="grey", lw=0.6)
    ax.set_title(d["pais"].iloc[0])

# Un referente, un caso de diagnóstico y dos casos de contraste (se pueden cambiar)
ejemplos = ["GBR", "CHE", "CHN", "COL"]
fig, axes = plt.subplots(1, len(ejemplos), figsize=(16, 4))
for iso, ax in zip(ejemplos, axes):
    trayectoria(iso, ax)
axes[0].set_ylabel("Índice (1995 = 100)")
axes[0].legend(fontsize=8)
fig.suptitle("Trayectorias 1995–2023: PIB frente a CO₂ territorial y de consumo", y=1.03)
plt.tight_layout()
fig.savefig(FIGURAS / "04_trayectorias.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Desagregación, robustez y sesgos

### 11.1 ¿Cambia el resultado al desagregar? (Sesión 7: Simpson; Sesión 11: pregunta 6)

Se divide a los países en tres grupos de ingreso (terciles del PIB per cápita promedio 2019–2023) y según si son importadores o exportadores netos de emisiones en 2023. Esto también pone a prueba la hipótesis.

In [ ]:
pib_pc_final = panel[panel["anio"].between(*FINAL)].groupby("iso3")["pib_pc"].mean()
balance_2023 = panel[panel["anio"] == 2023].set_index("iso3")["balance_comercio"]

desag = con_crecimiento.assign(
    grupo_ingreso=pd.qcut(pib_pc_final.reindex(con_crecimiento.index), 3,
                          labels=["Ingreso bajo", "Ingreso medio", "Ingreso alto"]),
    balance_2023=balance_2023.reindex(con_crecimiento.index),
)

def resumen_grupo(g):
    return pd.Series({
        "países": len(g),
        "cambian de clase": int(g["cambia_de_clase"].sum()),
        "% que cambia": g["cambia_de_clase"].mean() * 100,
        "absoluto territorial": int((g["clase_territorial"] == "Desacoplamiento absoluto").sum()),
        "absoluto consumo": int((g["clase_consumo"] == "Desacoplamiento absoluto").sum()),
    })

print("Por grupo de ingreso:")
print(desag.groupby("grupo_ingreso", observed=True)[["cambia_de_clase", "clase_territorial", "clase_consumo"]].apply(resumen_grupo).round(1).to_string())
print("\nPor balance de emisiones en 2023:")
print(desag.groupby("balance_2023")[["cambia_de_clase", "clase_territorial", "clase_consumo"]].apply(resumen_grupo).round(1).to_string())

In [ ]:
# ¿Qué tipo de cambio ocurre en cada grupo de ingreso?
tipo_cambio = desag[desag["cambia_de_clase"]].assign(
    cambio=lambda d: d["clase_territorial"].str.replace("Desacoplamiento ", "") + " → " + d["clase_consumo"].str.replace("Desacoplamiento ", "")
)
pd.crosstab(tipo_cambio["cambio"], tipo_cambio["grupo_ingreso"])

**Lectura:** el desacoplamiento absoluto se concentra en países de ingreso alto, y es ahí donde contar el comercio "quita" referentes. En los países de ingreso bajo los cambios de clase son más frecuentes, pero casi todos ocurren entre *relativo* y *sin desacoplamiento*: pasar a la contabilidad de consumo no cambia si están reduciendo emisiones, sino qué tan rápido crecen frente al PIB. Mirar solo el total escondería esta diferencia.

### 11.2 ¿Cambia la interpretación con valores per cápita o por unidad de PIB?

- **Per cápita:** misma clasificación con PIB per cápita y CO₂ per cápita (quita el efecto del crecimiento de la población).
- **Intensidad:** kg de CO₂ por dólar. Si cae, la economía es más eficiente en carbono; por sí solo eso implica desacoplamiento relativo, no absoluto.

In [ ]:
cambios_pc = tabla_cambios(INICIO, FINAL, columnas=("pib_pc", "co2_terr_pc", "co2_cons_pc"))
cambios_pc["clase_territorial_pc"] = [clasificar(p, c) for p, c in zip(cambios_pc["g_pib_pc"], cambios_pc["g_co2_terr_pc"])]
cambios_pc["clase_consumo_pc"] = [clasificar(p, c) for p, c in zip(cambios_pc["g_pib_pc"], cambios_pc["g_co2_cons_pc"])]
cambios_pc = marcar_sin_dato(cambios_pc, INICIO, "clase_consumo_pc")
validos_pc = cambios_pc.loc[cambios_pc.index.isin(validos.index)]

pd.DataFrame({
    "Totales · territorial": validos["clase_territorial"].value_counts(),
    "Totales · consumo": validos["clase_consumo"].value_counts(),
    "Per cápita · territorial": validos_pc["clase_territorial_pc"].value_counts(),
    "Per cápita · consumo": validos_pc["clase_consumo_pc"].value_counts(),
}).reindex(CLASES).fillna(0).astype(int)

In [ ]:
intensidad = tabla_cambios(INICIO, FINAL, columnas=("intensidad_terr", "intensidad_cons"))
intensidad = intensidad[intensidad.index.isin(validos.index)]
print("Países cuya intensidad territorial cayó:", int((intensidad["g_intensidad_terr"] < 0).sum()), "de", len(intensidad))
print("Países cuya intensidad de consumo cayó:", int((intensidad["g_intensidad_cons"] < 0).sum()), "de", len(intensidad))
print(f"Mediana del cambio de intensidad: territorial {intensidad['g_intensidad_terr'].median():.1f} % | consumo {intensidad['g_intensidad_cons'].median():.1f} %")

### 11.3 Robustez: ¿se sostiene con otro período inicial?

In [ ]:
def resumen_periodo(inicio, final):
    c = tabla_cambios(inicio, final)
    c["t"] = [clasificar(p, x) for p, x in zip(c["g_pib"], c["g_co2_terr_mt"])]
    c["k"] = [clasificar(p, x) for p, x in zip(c["g_pib"], c["g_co2_cons_mt"])]
    c = marcar_sin_dato(c, inicio, "k")
    c = c[c["k"] != SIN_DATO]
    crece = c[c["t"] != "PIB sin crecimiento"]
    return {
        "período": f"{inicio[0]}–{inicio[1]} vs {final[0]}–{final[1]}",
        "países con crecimiento": len(crece),
        "absoluto territorial": int((crece["t"] == "Desacoplamiento absoluto").sum()),
        "absoluto consumo": int((crece["k"] == "Desacoplamiento absoluto").sum()),
        "absoluto en ambas": int(((crece["t"] == "Desacoplamiento absoluto") & (crece["k"] == "Desacoplamiento absoluto")).sum()),
        "cambian de clase": int((crece["t"] != crece["k"]).sum()),
    }

pd.DataFrame([resumen_periodo(INICIO, FINAL), resumen_periodo((2005, 2009), FINAL)]).set_index("período")

### 11.4 Sesgos y uso responsable (Sesión 11)

Una base limpia no es una base sin sesgos. Auditoría de las conclusiones con las siete preguntas de la Sesión 11:

In [ ]:
pib_paises = gdp[gdp["anio"] == 2023]
cobertura = pib_paises.loc[pib_paises["iso3"].isin(validos.index), "pib"].sum() / pib_paises["pib"].sum()

print(f"Países con PIB en 2023: {len(pib_paises)} | clasificados con ambas contabilidades: {len(validos)}")
print(f"Participación de esos países en el PIB mundial de 2023: {cobertura:.1%}")

excluidos = pib_paises[~pib_paises["iso3"].isin(validos.index)].nlargest(10, "pib")
nombres = pib.drop_duplicates("Code").set_index("Code")["Entity"]
print("\nPaíses excluidos con mayor PIB:", ", ".join(nombres.reindex(excluidos["iso3"]).tolist()))

| Pregunta | Respuesta para este proyecto |
|---|---|
| 1. ¿Quiénes integran la población de interés? | Todos los países que buscan crecer reduciendo emisiones. |
| 2. ¿Quiénes pueden entrar a la fuente? | Solo países con datos de comercio suficientes para estimar emisiones de consumo (~120). |
| 3. ¿Quiénes están subrepresentados? | Países pequeños y de bajos ingresos, sobre todo de África y Asia Central. Los clasificados concentran casi todo el PIB mundial, pero no todos los países. **Sesgo de selección.** |
| 4. ¿La variable mide lo mismo para todos? | No del todo: las emisiones de consumo se estiman con modelos de comercio, con más incertidumbre en economías pequeñas, y en cinco países los primeros años son copias de la territorial. **Sesgo de medición.** |
| 5. ¿Hay faltantes sistemáticos o duplicaciones? | Los faltantes de consumo se concentran en países pobres; no hay duplicados. |
| 6. ¿Cómo cambia el resultado al desagregar? | Mucho: el desacoplamiento absoluto es casi exclusivo del ingreso alto (sección 11.1). |
| 7. ¿Qué decisión puede afectar a quién? | Señalar a un país como "referente" o "en diagnóstico" puede influir en su reputación climática. Por eso se presentan como **casos para estudiar**, no como juicios, y no se interpreta la falta de desacoplamiento en países pobres como un fracaso: parten de emisiones per cápita muy bajas. |

## 12. Hallazgos

Cada hallazgo sigue el formato de la Sesión 7: hallazgo, evidencia, interpretación, contexto y nueva pregunta.

In [ ]:
print(f"1. Cambian de clase: {n_cambian} de {len(con_crecimiento)} países ({n_cambian / len(con_crecimiento):.0%})")
print(f"2. Desacoplamiento absoluto: {abs_terr} (territorial) → {abs_cons} (consumo); referentes en ambas: {len(referentes)}")
print(f"3. Requieren diagnóstico: {len(diagnostico_adicional)} → {', '.join(diagnostico_adicional['pais'])}")
por_ingreso = desag.groupby("grupo_ingreso", observed=True)[["cambia_de_clase", "clase_territorial", "clase_consumo"]].apply(resumen_grupo)
print(f"4. Absoluto territorial por ingreso (bajo/medio/alto): {por_ingreso['absoluto territorial'].astype(int).tolist()}")
print(f"5. Cobertura: {len(validos)} países, {cobertura:.0%} del PIB mundial de 2023")

**Hallazgo 1. Uno de cada cuatro países cambia de clase al contar el comercio.**
- *Evidencia:* 26 de 112 países con crecimiento del PIB (23 %) tienen una clase distinta con emisiones de consumo; con otro período inicial (2005–2009) el resultado es similar.
- *Interpretación:* la contabilidad territorial por sí sola no basta para identificar desacoplamiento.
- *Nueva pregunta:* ¿qué sectores explican las emisiones importadas de los países que cambian?

**Hallazgo 2. Contar el comercio reduce el número de casos de desacoplamiento absoluto.**
- *Evidencia:* 34 países lo logran con la contabilidad territorial y 26 con la de consumo; 25 lo logran con ambas (referentes).
- *Interpretación:* los referentes, en su mayoría europeos de ingreso alto, redujeron emisiones sin importar cómo se midan.

**Hallazgo 3. Nueve países requieren diagnóstico adicional.**
- *Evidencia:* reducen su CO₂ territorial pero su CO₂ de consumo crece (por ejemplo Suiza: −22 % territorial, +25 % consumo).
- *Interpretación:* es compatible con trasladar producción al exterior, pero los datos no lo demuestran (análisis descriptivo).
- *Nueva pregunta:* ¿de qué países provienen sus importaciones de emisiones?

**Hallazgo 4. El desacoplamiento absoluto se concentra en países de ingreso alto.**
- *Evidencia:* 24 de 37 países de ingreso alto lo logran con la contabilidad territorial, frente a 2 de 38 de ingreso bajo.
- *Contexto:* los países de ingreso bajo parten de emisiones per cápita muy bajas; que sus emisiones crezcan no es comparable con las de economías ricas.

**Hallazgo 5. La hipótesis se cumple en parte.**
- *Evidencia:* los 8 países que pierden el desacoplamiento absoluto al contar el comercio son importadores netos de emisiones (32 → 24); entre los exportadores netos no cambia (2 → 2). Sin embargo, el porcentaje total de países que cambia de clase es parecido en ambos grupos (24 % frente a 20 %).
- *Interpretación:* ser importador neto se asocia con perder el desacoplamiento *absoluto*, pero no con cambiar de clase en general.

**Limitación clave para el dashboard:** los países clasificados concentran el 96 % del PIB mundial, pero quedan fuera muchos países pequeños y de bajos ingresos sin datos de consumo.

## 13. Exportación para el dashboard (Sesión 14)

El dashboard se alimenta de los datos integrados, no de las fuentes por separado.

| Indicador | Fórmula | Unidad de análisis | Unidad |
|---|---|---|---|
| Países que cambian de clase | Países con clase territorial ≠ clase de consumo / países con ambas clases y PIB creciente | país | % |
| Desacoplamiento absoluto | Países con PIB creciente y CO₂ decreciente | país | número |
| Cambio del PIB | Promedio 2019–2023 / promedio 1995–1999 − 1 | país | % |
| Cambio del CO₂ (territorial / consumo) | Ídem con emisiones totales | país | % |
| Brecha consumo − territorial | (consumo − territorial) / territorial | país-año | % |

**Fecha de corte:** datos hasta 2023 (último año común), consultados el 22/11/2026.

In [ ]:
panel.sort_values(["iso3", "anio"]).to_csv(PROCESSED / "panel_integrado.csv", index=False)

clasificacion = (cambios
                 .join(cambios_pc[["g_pib_pc", "g_co2_terr_pc", "g_co2_cons_pc", "clase_territorial_pc", "clase_consumo_pc"]])
                 .join(desag[["grupo_ingreso", "balance_2023"]]))
clasificacion.index.name = "iso3"
clasificacion.reset_index().to_csv(PROCESSED / "clasificacion_paises.csv", index=False)

print(f"data/processed/panel_integrado.csv: {len(panel)} filas (país-año)")
print(f"data/processed/clasificacion_paises.csv: {len(clasificacion)} filas (país)")
print("Figuras guardadas:", sorted(p.name for p in FIGURAS.glob("*.png")))

## 14. Limitaciones, próximos pasos y fuentes

### Limitaciones
1. **Cobertura:** solo ~120 países tienen emisiones de consumo; quedan fuera muchos países pequeños y de bajos ingresos (sesgo de selección). Taiwán y Venezuela quedan fuera por falta de PIB.
2. **Alcance de las emisiones:** solo CO₂ fósil e industrial. No incluye cambio de uso del suelo, aviación y navegación internacional, ni otros gases de efecto invernadero.
3. **Emisiones de consumo estimadas:** provienen de modelos de comercio y tienen más incertidumbre que las territoriales. En Bélgica, Luxemburgo, Sudáfrica, Namibia y Botsuana la serie de los años 90 es una copia de la territorial, por lo que no se clasificaron con consumo. Noruega no tiene consumo antes de 2003.
4. **Definición de períodos:** la clasificación depende de las ventanas elegidas (ver robustez).
5. **Sin causalidad:** el análisis es descriptivo; un cambio de clase es compatible con el traslado de producción al exterior, pero no lo demuestra.
6. **Fuente intermedia:** las cuatro series se descargaron de Our World in Data; se citan los productores originales.

### Próximos pasos
1. Construir el dashboard en Streamlit (`app.py`) con `data/processed/` y plotly, con la pregunta, los indicadores, la tabla cruzada, los casos para el comité y la limitación de cobertura.
2. Completar `README.md` y `requirements.txt`, y publicar el repositorio en GitHub.
3. Preparar la presentación ejecutiva de 10 minutos.

### Fuentes
- Global Carbon Budget (2026); Population based on various sources (2024) – with major processing by Our World in Data. *CO₂ emissions per capita* y *Per capita consumption-based CO₂ emissions*. https://ourworldindata.org/co2-emissions
- Eurostat, OECD, IMF, and World Bank (2026) – with minor processing by Our World in Data. *Gross domestic product (GDP)*. https://ourworldindata.org/grapher/gdp-worldbank
- HYDE (2023); Gapminder (2022); UN WPP (2024) – with major processing by Our World in Data. *Population*. https://ourworldindata.org/grapher/population